In [ ]:
set.seed(1)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(tidyverse)
library(patchwork)
library(harmony)
library(readxl)
library(viridis)

In [ ]:
date <- "2026-06-05_new_adult"

In [ ]:
data_dir <- "/project/work/2026-05-25_create_summary/Data/"

# 准备数据

In [ ]:
obj_add <- readRDS("/project/work/2026-05-25_create_summary/Data/snRNA_30_32years_raw_counts.rds")

In [ ]:
new_df <- readRDS("/project/work/2026-05-25_create_summary/Data/2026-05-13_human_rna_gene_name.rds")
current_genes <- rownames(obj_add)

match_idx <- match(current_genes, new_df$GeneName)
new_gene_names <- new_df$hgnc_symbol[match_idx]

is_missing <- is.na(new_gene_names) | new_gene_names == ""
new_gene_names[is_missing] <- current_genes[is_missing]

new_gene_names <- make.unique(new_gene_names)

In [ ]:
counts_matrix <- GetAssayData(obj_add, assay = "RNA", layer = "counts")
head(counts_matrix)

In [ ]:
rownames(counts_matrix) <- new_gene_names

In [ ]:
obj_add[["RNA"]] <- CreateAssayObject(counts = counts_matrix)

In [ ]:
counts_matrix <- GetAssayData(obj_add, assay = "RNA", layer = "counts")
head(counts_matrix)

In [ ]:
head(rownames(counts_matrix),50)
tail(rownames(counts_matrix),50)

In [ ]:
obj <- readRDS("/project/work/2026-05-25_create_summary/Data/2026-06-05_adult_human_harmony_anno.rds")

In [ ]:
obj <- merge(obj,obj_add)
obj

In [ ]:
obj[["RNA"]] <- JoinLayers(obj[["RNA"]])

# QC

In [ ]:
obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 5)
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA","percent.mt"), ncol = 4, group.by = "donor_id", pt.size = 0)

In [ ]:
dim(obj)
obj <- obj[,obj$percent.mt < 20]
dim(obj)

# dim(obj)
# obj <- obj[,obj$nCount_RNA > 500]
# dim(obj)

# dim(obj)
# obj <- obj[,obj$nFeature_RNA > 300]
# dim(obj)

In [ ]:
dim(obj)
qs <- quantile(obj$nFeature_RNA, probs = c(0.05, 0.95), na.rm = TRUE)
qs
# obj <- subset(obj,subset = nFeature_RNA >= qs[[1]] & nFeature_RNA <= qs[[2]])
dim(obj)

# Process analysis

In [ ]:
obj[["RNA"]] <- split(obj[["RNA"]], f = obj$donor_id)

In [ ]:
obj <- NormalizeData(obj, normalization.method = "LogNormalize", scale.factor = 10000)

In [ ]:
obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 3000)

In [ ]:
obj <- ScaleData(obj, features = VariableFeatures(obj))

In [ ]:
obj <- RunPCA(obj, features = VariableFeatures(obj), npcs = 40)
obj <- FindNeighbors(obj, dims = 1:20, reduction = "pca")

In [ ]:
obj <- FindClusters(obj, resolution = 0.4, cluster.name = "unintegrated_clusters04")
obj <- RunUMAP(obj, dims = 1:20, reduction = "pca", reduction.name = "umap.unintegrated")
# DimPlot(obj, reduction = "umap.unintegrated")

In [ ]:
saveRDS(obj,file = paste0(data_dir, date, "_human_unintegrated.rds"))

In [ ]:
paste0(data_dir, date, "_human_unintegrated.rds")

# harmony

In [ ]:
obj <- IntegrateLayers(object = obj, method = HarmonyIntegration, orig.reduction = "pca", new.reduction = "harmony", verbose = FALSE)

In [ ]:
obj[["RNA"]] <- JoinLayers(obj[["RNA"]])

In [ ]:
obj <- FindNeighbors(obj, reduction = "harmony", dims = 1:20)
obj <- FindClusters(obj, resolution = 0.4, cluster.name = "harmony_clusters04")

In [ ]:
obj <- RunUMAP(obj, reduction = "harmony", dims = 1:20, reduction.name = "umap.harmony")

In [ ]:
saveRDS(obj,file = paste0(data_dir, date, "_human_harmony.rds"))

In [ ]:
paste0(data_dir, date, "_human_harmony.rds")

# read

In [ ]:
obj <- readRDS('/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony.rds')
obj

In [ ]:
obj$tissue_age <- paste0(obj$tissue,obj$donor_age)

In [ ]:
options(repr.plot.width = 13, repr.plot.height =8)
DimPlot(obj, reduction = "umap.unintegrated", group.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.unintegrated", group.by = "celltype",raster=FALSE)

In [ ]:
options(repr.plot.width = 13, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =6)
DimPlot(obj, reduction = "umap.harmony", group.by = "celltype", label= TRUE,raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =6)
DimPlot(obj, reduction = "umap.harmony", group.by = "harmony_clusters04",raster=FALSE)

In [ ]:
CONTACT_EMAIL <- CONTACT_EMAIL[,c('orig.ident','nCount_RNA','nFeature_RNA','percent.mt','assay','disease',
                                  'donor_id','donor_age','sex','self_reported_ethnicity','tissue',
                                  'cell_type','majorclass',#'subclass',
                                  # 'doublet_info',
                                  'unintegrated_clusters04','seurat_clusters','harmony_clusters04')]

In [ ]:
obj$tissue <- as.character(obj$tissue)

In [ ]:
obj$tissue[obj$tissue == 'fovea centralis'] = 'fovea'
obj$tissue[obj$tissue == 'macula lutea'] = 'macula'
obj$tissue[obj$tissue == 'peripheral region of retina'] = 'peripheral'

In [ ]:
obj$donor_age <- gsub(" ", "", obj$donor_age)

In [ ]:
obj$tissue_age <- paste0(obj$tissue, "_", obj$donor_age)

In [ ]:
options(repr.plot.width = 50, repr.plot.height =6)
DimPlot(obj, reduction = "umap.unintegrated", group.by = 'unintegrated_clusters04', split.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 50, repr.plot.height =6)
DimPlot(obj, reduction = "umap.harmony", group.by = 'harmony_clusters04', split.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 5)
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA","percent.mt"), ncol = 3, group.by = "tissue_age", pt.size = 0)

# anno

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "harmony_clusters04", label = TRUE,raster=FALSE)

In [ ]:
obj$celltype <- paste0('UN_',obj$harmony_clusters04)

In [ ]:
obj$celltype[obj$harmony_clusters04 %in% c('11','10','0')] = 'Rod'
obj$celltype[obj$harmony_clusters04 %in% c('38','37','35','3')] = 'Cone'
obj$celltype[obj$harmony_clusters04 %in% c('8')] = 'HC'
obj$celltype[obj$harmony_clusters04 %in% c('39','1','12','21')] = 'RGC'
obj$celltype[obj$harmony_clusters04 %in% c('14','27','34','17','22','18','26','28','6','4','7')] = 'BC'
obj$celltype[obj$harmony_clusters04 %in% c('25','24','31','9','5','33','20','23','15','13','19','16')] = 'AC'
obj$celltype[obj$harmony_clusters04 %in% c('2','36')] = 'MG'
obj$celltype[obj$harmony_clusters04 %in% c('32')] = 'Micro'
obj$celltype[obj$harmony_clusters04 %in% c('29')] = 'Astro'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6)
DimPlot(obj, reduction = "umap.harmony", group.by = "celltype", label = TRUE,raster=FALSE)

In [ ]:
use.markers <- c(
  "RHO", "PDE6A", "CNGA1", "SAG", "NR2E3", "GNAT1",              # Rod
  "OPN1MW", "OPN1LW", "OPN1SW", "GNAT2", "PDE6H", "ARR3", "GUCA1C", # Cone
  "VSX1", "VSX2", "GRM6", "TRPM1", "PRKCA", "CABP5",              # BC
  "GAD1", "GAD2", "SLC6A9", "TFAP2A", "TFAP2B",                    # AC
  "ONECUT1", "ONECUT2", "LHX1", "CALB1", "PROX1",                  # HC
  "RBPMS", "POU4F1", "POU4F2", "SLC17A6", "NEFL", "NEFM", "SNCG", "GAP43",  # RGC
  "RLBP1", "GLUL", "RGR", "DKK3", #"AQP4", "SOX9", "S100B", "VIM",  # MG
  "RPE65",
  "AIF1", "C1QA", "C1QB", "CX3CR1", "P2RY12", "TMEM119",          # Microglia
  "GFAP", "ALDH1L1", "GJA1"                                         # Astrocyte
)

length(unique(use.markers))

In [ ]:
use.types <- c(
  rep("Rod", 6),
  rep("Cone", 7),
  rep("BC", 6),
  rep("AC", 5),
  rep("HC", 5),
  rep("RGC", 7),
  rep("MG", 5),
  rep("RPE", 1),
  rep("Micro", 6),
  rep("Astrocyte", 3)
)

length(use.types)

In [ ]:
use.markers.df <- data.frame(features.plot = use.markers, type = use.types)

In [ ]:
p <- DotPlot(obj, features = use.markers, group.by = "celltype")+
scale_colour_gradient2(low = "black", mid = "grey", high = "red")+ 
theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"))

In [ ]:
use.data <- p$data

In [ ]:
use.data <- merge(use.data, use.markers.df, by = "features.plot")

In [ ]:
use.data$features.plot <- factor(use.data$features.plot, levels = use.markers)

In [ ]:
use.data$type <- factor(use.data$type, levels = unique(use.types))

In [ ]:
p1 <- ggplot(use.data, aes(x = features.plot, y = id)) +
  geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +  # 使用点大小和颜色表示信息
  scale_size(range = c(0.1, 5)) +  # 调整点的大小范围
  scale_color_gradient2(low = "black", mid = "grey", high = "red") +  # 定义颜色梯度
  facet_wrap(~ type, scales = "free_x", nrow = 1) +  # 根据类型分面
  theme_classic() +
  theme(text = element_text(color = "black"),
    axis.text.y = element_text(color = "black", size = 12),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 10),
    legend.title = element_text(size = 10),  # 图例标题字号及下方间距
    strip.text = element_text(size = 11)  # 调整分面标签字体大小
  ) +
  labs(x = "Features", y = "Cell Type", size = "Percentag\n Expressed", color = "Scaled\nExpression")

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8)
p1

# save

In [ ]:
obj <- subset(obj,celltype!="UN_30")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6)
DimPlot(obj, reduction = "umap.harmony", group.by = "celltype", label = TRUE,raster=FALSE)

In [ ]:
obj$donor_age <- factor(obj$donor_age,level = c('21years','25years','30years','32years','67years','70years','75years','86years'))
obj$celltype <- factor(obj$celltype,level = c('Rod','Cone','AC','BC','HC','MG','RGC','Micro','Astro'))
obj$tissue <- factor(obj$tissue,level = c('fovea','macula','peripheral'))

In [ ]:
obj$tissue_age <- factor(obj$tissue_age,level = c('fovea_21years','macula_21years','peripheral_21years',
                                                 'fovea_25years','macula_25years','peripheral_25years',
                                                  'macula_30years','peripheral_30years',
                                                  'fovea_32years','macula_32years','peripheral_32years',
                                                 'fovea_67years','peripheral_67years',
                                                 'macula_70years',
                                                 'macula_75years','peripheral_75years',
                                                 'macula_86years','peripheral_86years'))

In [ ]:
saveRDS(obj,paste0(data_dir, date, "_human_harmony_anno.rds")) # 后更名为2026-06-05_adult_human_harmony_anno.rds

In [ ]:
paste0(data_dir, date, "_human_harmony_anno.rds")